# Python Programming for Chemists: Visualisation & Simulation Examples 

You will need the Python libraries [ase](https://wiki.fysik.dtu.dk/ase/) and [matplotlib](https://matplotlib.org/).

Local uv project: `uv add ase matplotlib` (online, e.g. Colab: `pip install ase matplotlib`)

**Note**: Its not expected that you understand in detail what is happening in this notebook yet. Its merely a demonstration of what can be done with a few lines of Python code.  
However, you will understand it at the end of the lecture!


In [ ]:
!pip install ase matplotlib

###  Molecular Dynamics Simulation of Acetonitrile


A [molecular dynamics](https://en.wikipedia.org/wiki/Molecular_dynamics#:~:text=Molecular%20dynamics%20(MD)%20is%20a,%22evolution%22%20of%20the%20system.) (MD) simulates atomic and molecular movements over time by solving Newton's equations of motion, using force fields to calculate interactions.   
It is widely used in chemical physics, materials science, and biophysics.


In [ ]:
# load the necessary functions and modules
import ase.units as units
import matplotlib.pyplot as plt
from ase.calculators.acn import ACN
from ase.constraints import FixLinearTriatomic
from ase.io import Trajectory, read
from ase.md import Langevin
from ase.visualize.plot import animate
from IPython.display import HTML

def show_system(traj):
    """Play the simulation. Carbon is gray, nitrogen is blue."""
    fig, ax = plt.subplots(figsize=(5, 5))
    movie = animate(traj, ax=ax, interval=80, rotation="15x,15y")
    html = HTML(movie.to_jshtml())
    plt.close(fig)
    return html

def print_energy(atoms):
    """Print potential energy, kinetic energy, and temperature."""
    epot = atoms.get_potential_energy()
    ekin = atoms.get_kinetic_energy()
    temperature = ekin / len(atoms) / (1.5 * units.kB)
    print(f"Potential Energy: {epot:.3f}  Kinetic Energy: {ekin:.3f}  Temperature: {temperature:.1f}")


In [ ]:
!cat ../data/acn.xyz

In [ ]:
# read the starting coordinates
atoms = read("../data/acn.xyz")
tag = 'acn'

# cubic box, side length in Angstrom. Repeat the box in every direction.
atoms.set_cell([13.34, 13.34, 13.34])
atoms.pbc = True

# 27 molecules. Each molecule is three sites: CH3, C, N
atoms.set_masses([15.035, 12.011, 14.007] * 27)
# keep each CH3-C-N molecule stiff, so the bonds do not vibrate
triples = []
for i in range(27):
    triples.append((3 * i, 3 * i + 1, 3 * i + 2))
atoms.constraints = FixLinearTriatomic(triples=triples)


In [ ]:
%%time
# define the molecular dynamics physics / rules / model
atoms.calc = ACN(rc=6.6)
# define the simulation details, temperature, time steps, parameters, logfile name
md = Langevin(atoms, 1 * units.fs,
              temperature_K=300,
              friction=0.01,
              logfile=tag + '.log')

# define a file to save the generated structures during the dynamics
traj = Trajectory(tag + '.traj', 'w', atoms)
md.attach(traj.write, interval=10)
md.attach(print_energy,interval=10,atoms=atoms)

# start the simulation for n time steps
n_steps = 500
md.run(n_steps)

In [ ]:
# play the simulation
traj = Trajectory("acn.traj")
show_system(traj)

### Optional Exercise

Play around with the code and the parameters:

* Find the temperature parameter. Set the temperature to 100 K and observe the result.

* How would you change the code to make much longer molecular dynamics runs?

* Remove the bond constraints holding the molecules together